# Merge AutoScout24 primary and secondary CSV files

This notebook loads the matching primary CSV and `_secondary.csv`, merges them on `listing_url`, keeps the primary row base, appends secondary-only columns, fills missing overlapping values when possible, and saves a `fullRaw_...csv` output.

Example output name:
- `scrape_audi_q4_20260423.csv`
- `scrape_audi_q4_20260423_secondary.csv`
- `fullRaw_scrape_audi_q4_20260423.csv`

In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

PRIMARY_CSV_PATH = None  # Example: Path("scrape_audi_q4_20260423.csv")
SECONDARY_CSV_PATH = None  # Example: Path("scrape_audi_q4_20260423_secondary.csv")
PRIMARY_GLOB = "scrape_audi_q4_*.csv"
SECONDARY_SUFFIX = "_secondary.csv"
MERGE_KEY = "listing_url"


In [ ]:
def find_latest_primary_csv():
    candidates = [
        path for path in Path.cwd().glob(PRIMARY_GLOB)
        if path.is_file() and not path.name.endswith(SECONDARY_SUFFIX) and not path.name.startswith("fullRaw_")
    ]
    if not candidates:
        raise FileNotFoundError("No primary CSV file matching scrape_audi_q4_*.csv was found.")
    return max(candidates, key=lambda path: path.stat().st_mtime)


def find_matching_secondary_csv(primary_csv_path):
    primary_csv_path = Path(primary_csv_path)
    secondary_name = f"{primary_csv_path.stem}_secondary.csv"
    secondary_path = primary_csv_path.with_name(secondary_name)
    if not secondary_path.exists():
        raise FileNotFoundError(f"Matching secondary CSV was not found: {secondary_path}")
    return secondary_path


def resolve_input_paths(primary_csv_path=None, secondary_csv_path=None):
    primary_path = Path(primary_csv_path) if primary_csv_path else find_latest_primary_csv()
    secondary_path = Path(secondary_csv_path) if secondary_csv_path else find_matching_secondary_csv(primary_path)
    return primary_path, secondary_path


def build_output_path(primary_csv_path, secondary_csv_path):
    primary_csv_path = Path(primary_csv_path)
    secondary_csv_path = Path(secondary_csv_path)

    primary_stem = primary_csv_path.stem
    secondary_stem = secondary_csv_path.stem.replace("_secondary", "")
    base_stem = primary_stem if primary_stem == secondary_stem else secondary_stem

    return primary_csv_path.with_name(f"fullRaw_{base_stem}.csv")


def validate_merge_key(dataframe, dataframe_name):
    if MERGE_KEY not in dataframe.columns:
        raise KeyError(f"{dataframe_name} does not contain the merge key: {MERGE_KEY}")


def deduplicate_on_merge_key(dataframe, dataframe_name):
    duplicate_mask = dataframe.duplicated(subset=[MERGE_KEY], keep="first")
    duplicate_count = int(duplicate_mask.sum())
    if duplicate_count:
        print(f"{dataframe_name}: dropping {duplicate_count} duplicate rows by {MERGE_KEY}.")
        dataframe = dataframe.loc[~duplicate_mask].copy()
    return dataframe


def merge_primary_and_secondary(primary_df, secondary_df):
    validate_merge_key(primary_df, "Primary CSV")
    validate_merge_key(secondary_df, "Secondary CSV")

    primary_df = deduplicate_on_merge_key(primary_df, "Primary CSV")
    secondary_df = deduplicate_on_merge_key(secondary_df, "Secondary CSV")

    primary_indexed = primary_df.set_index(MERGE_KEY)
    secondary_indexed = secondary_df.set_index(MERGE_KEY)

    primary_columns = list(primary_indexed.columns)
    secondary_columns = list(secondary_indexed.columns)

    overlapping_columns = [
        column for column in secondary_columns
        if column in primary_columns
    ]
    secondary_only_columns = [
        column for column in secondary_columns
        if column not in primary_columns
    ]

    merged_indexed = primary_indexed.copy()

    for column in overlapping_columns:
        merged_indexed[column] = merged_indexed[column].combine_first(secondary_indexed[column])

    if secondary_only_columns:
        merged_indexed = merged_indexed.join(secondary_indexed[secondary_only_columns], how="left")

    ordered_columns = primary_columns + secondary_only_columns
    merged_df = merged_indexed.reset_index().reindex(columns=[MERGE_KEY] + ordered_columns)

    return merged_df, overlapping_columns, secondary_only_columns


def run_merge(primary_csv_path, secondary_csv_path):
    primary_df = pd.read_csv(primary_csv_path)
    secondary_df = pd.read_csv(secondary_csv_path)

    merged_df, overlapping_columns, secondary_only_columns = merge_primary_and_secondary(
        primary_df,
        secondary_df,
    )

    output_path = build_output_path(primary_csv_path, secondary_csv_path)
    merged_df.to_csv(output_path, index=False, encoding="utf-8")

    return {
        "primary_rows": len(primary_df),
        "secondary_rows": len(secondary_df),
        "merged_rows": len(merged_df),
        "overlapping_columns": overlapping_columns,
        "secondary_only_columns": secondary_only_columns,
        "output_path": output_path,
        "merged_df": merged_df,
    }


In [ ]:
primary_csv_path, secondary_csv_path = resolve_input_paths(PRIMARY_CSV_PATH, SECONDARY_CSV_PATH)

print(f"Primary CSV:   {primary_csv_path}")
print(f"Secondary CSV: {secondary_csv_path}")

result = run_merge(primary_csv_path, secondary_csv_path)

print(f"Primary rows:  {result['primary_rows']}")
print(f"Secondary rows:{result['secondary_rows']}")
print(f"Merged rows:   {result['merged_rows']}")
print(f"Output file:   {result['output_path']}")
print(f"Overlapping columns merged with combine_first: {len(result['overlapping_columns'])}")
print(f"Secondary-only columns appended: {len(result['secondary_only_columns'])}")

result['merged_df'].head()